# Семинар: метрики и KNN

In [17]:
import math 

## 1. Данные

### Квартиры: задача регрессии
Каждая квартира - это **объект**. У неё 3 **признака**: площадь (м²), этаж и сколько минут идти до метро.
**Ответ** - цена в миллионах рублей.

In [18]:
# Признаки: [площадь, этаж, до метро (мин)]
X_flats = [
    [42, 5, 10], [65, 12, 4], [30, 2, 18], [55, 7, 8],
    [80, 15, 5], [38, 3, 15], [70, 9, 6],  [48, 4, 12],
    [45, 6, 9],  [60, 10, 7], [35, 1, 20], [75, 14, 3],
]
# Ответы: цена, млн руб.
y_flats = [8.1, 12.4, 5.6, 10.2, 15.0, 6.9, 13.1, 8.8, 8.6, 11.5, 6.0, 14.6]

print(f"Объектов: {len(X_flats)}, признаков у каждого: {len(X_flats[0])}")
print()
print(f"{'№':>3} | {'Площадь':>7} | {'Этаж':>4} | {'Метро':>5} | {'Цена':>5}")
print("-" * 38)
for i in range(len(X_flats)):
    area, floor, metro = X_flats[i]
    print(f"{i:>3} | {area:>7} | {floor:>4} | {metro:>5} | {y_flats[i]:>5}")

Объектов: 12, признаков у каждого: 3

  № | Площадь | Этаж | Метро |  Цена
--------------------------------------
  0 |      42 |    5 |    10 |   8.1
  1 |      65 |   12 |     4 |  12.4
  2 |      30 |    2 |    18 |   5.6
  3 |      55 |    7 |     8 |  10.2
  4 |      80 |   15 |     5 |  15.0
  5 |      38 |    3 |    15 |   6.9
  6 |      70 |    9 |     6 |  13.1
  7 |      48 |    4 |    12 |   8.8
  8 |      45 |    6 |     9 |   8.6
  9 |      60 |   10 |     7 |  11.5
 10 |      35 |    1 |    20 |   6.0
 11 |      75 |   14 |     3 |  14.6


### Фрукты: задача классификации
Признаки: вес в граммах и «гладкость» кожуры по шкале от 0 до 10. Ответ - класс: `'яблоко'` или `'груша'`.

In [19]:
# Признаки: [вес (г), гладкость (0–10)]
X_fruits = [
    [150, 8], [165, 9], [140, 7], [178, 8],
    [158, 5], [185, 4], [172, 3], [200, 5],
    [152, 9], [180, 4], [168, 8], [162, 4],
]
y_fruits = ['яблоко', 'яблоко', 'яблоко', 'яблоко',
            'груша',  'груша',  'груша',  'груша',
            'яблоко', 'груша',  'яблоко', 'груша']

for x, label in zip(X_fruits, y_fruits):
    print(f"вес {x[0]:>3} г, гладкость {x[1]:>2}  →  {label}")

вес 150 г, гладкость  8  →  яблоко
вес 165 г, гладкость  9  →  яблоко
вес 140 г, гладкость  7  →  яблоко
вес 178 г, гладкость  8  →  яблоко
вес 158 г, гладкость  5  →  груша
вес 185 г, гладкость  4  →  груша
вес 172 г, гладкость  3  →  груша
вес 200 г, гладкость  5  →  груша
вес 152 г, гладкость  9  →  яблоко
вес 180 г, гладкость  4  →  груша
вес 168 г, гладкость  8  →  яблоко
вес 162 г, гладкость  4  →  груша


### Обучающая и тестовая выборки
Модель учится на **train**, а проверяем её на **test** - на объектах, которых она не видела.
Первые 8 объектов отдадим в обучение, последние 4 - в тест.

In [20]:
X_flats_train, y_flats_train = X_flats[:8], y_flats[:8]
X_flats_test,  y_flats_test  = X_flats[8:], y_flats[8:]

X_fruits_train, y_fruits_train = X_fruits[:8], y_fruits[:8]
X_fruits_test,  y_fruits_test  = X_fruits[8:], y_fruits[8:]

print("Квартир в train:", len(X_flats_train), "| в test:", len(X_flats_test))
print("Фруктов в train:", len(X_fruits_train), "| в test:", len(X_fruits_test))

Квартир в train: 8 | в test: 4
Фруктов в train: 8 | в test: 4


## 2. Метрики регрессии

Пусть $y_i$ - правильный ответ, $a_i$ - прогноз модели, $n$ - число объектов.

**MAE** - средняя абсолютная ошибка:
$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n} |a_i - y_i|$$

**MSE** - средняя квадратичная ошибка:
$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n} (a_i - y_i)^2$$

**RMSE** - корень из MSE, чтобы вернуться к исходным единицам (рублям):
$$\text{RMSE} = \sqrt{\text{MSE}}$$

Пример: правда `[10, 6, 12]`, прогноз `[8, 7, 12]`. Ошибки: `−2, 1, 0`. MAE = (2 + 1 + 0) / 3 = 1.

In [21]:
def mae(y_true, y_pred):
    # y_true - список правильных ответов, y_pred - список прогнозов той же длины
    total = 0
    for i in range(len(y_true)):
        total += abs(y_pred[i] - y_true[i])   # abs - модуль числа
    return total / len(y_true)

print(mae([10, 6, 12], [8, 7, 12]))   # должно быть 1.0

1.0


### Задание 1. Функция MSE
Допишите функцию `mse` по аналогии с `mae`. Отличие одно: вместо модуля ошибки - её **квадрат** (`** 2`).

In [22]:
def mse(y_true, y_pred):
    return sum((y_pred[i] - y_true[i]) ** 2 for i in range(len(y_true))) / len(y_true)
    ...

In [23]:
try:
    assert mse([10, 6, 12], [8, 7, 12]) is not None, "функция ничего не возвращает - не забудьте return"
    assert abs(mse([10, 6, 12], [8, 7, 12]) - 5 / 3) < 1e-9, "для [10, 6, 12] и [8, 7, 12] ответ должен быть 5/3 ≈ 1.667"
    assert mse([0, 0], [3, 4]) == 12.5, "для [0, 0] и [3, 4] ответ должен быть (9 + 16) / 2 = 12.5"
    print("Верно!")
except Exception as e:
    print("Не верно(", e)

Верно!


Модель предсказала для тестовых квартир цены `[9.0, 11.0, 6.5, 16.0]`. Посчитайте **MSE** этих прогнозов на `y_flats_test` с помощью своей функции.

**Запишите ответ - число, округлённое до 3 знаков после запятой.**

In [24]:
predictions_1 = [9.0, 11.0, 6.5, 16.0]
answer_1 = ...  # ВАШ КОД ЗДЕСЬ: вызовите mse с правильными ответами и прогнозами
print("Ответ 1:")

Ответ 1:


RMSE - просто корень из MSE. Сравним метрики на двух моделях: вторая почти всегда точна, но **один раз сильно ошиблась**.

In [25]:
def rmse(y_true, y_pred):
    return math.sqrt(mse(y_true, y_pred))

y_true  = [8.6, 11.5, 6.0, 14.6]
model_a = [8.0, 11.0, 6.5, 14.0]   
model_b = [8.6, 11.5, 6.0, 24.6]  

print(f"{'':>8} | {'MAE':>6} | {'MSE':>6} | {'RMSE':>6}")
for name, pred in [('Модель А', model_a), ('Модель Б', model_b)]:
    print(f"{name:>8} | {mae(y_true, pred):>6.2f} | {mse(y_true, pred):>6.2f} | {rmse(y_true, pred):>6.2f}")

         |    MAE |    MSE |   RMSE
Модель А |   0.55 |   0.30 |   0.55
Модель Б |   2.50 |  25.00 |   5.00


**Обсудим:** почему у модели Б MSE выросла намного сильнее, чем MAE? Какую метрику выбрать, если одна большая ошибка для нас очень опасна?

## 3. Метрика классификации: accuracy

**Accuracy** - доля правильных ответов:
$$\text{Accuracy} = \frac{\text{число верных ответов}}{n} = \frac{1}{n}\sum_{i=1}^{n}[a_i = y_i]$$

Квадратные скобки $[a_i = y_i]$ означают: 1, если ответ верный, и 0, если нет.

### Задание 2. Функция accuracy
Посчитайте, сколько раз `y_pred[i] == y_true[i]`, и поделите на число объектов.

In [26]:
def accuracy(y_true, y_pred):
    correct = 0
    for i in range(len(y_true)):
        if y_true[i] == y_pred[i]:
            correct += 1
    return correct / len(y_true)

In [27]:
try:
    assert accuracy(['a', 'b', 'a'], ['a', 'b', 'a']) == 1.0, "если все ответы верны, должно быть 1.0"
    assert abs(accuracy(['a', 'b', 'a'], ['a', 'a', 'a']) - 2 / 3) < 1e-9, "2 верных из 3 - должно быть 0.667"
    assert accuracy(['кот', 'пёс'], ['пёс', 'кот']) == 0.0, "ни одного верного - должно быть 0.0"
    print("✅ Верно!")
except Exception as e:
    print("Не верно(", e)

✅ Верно!


### Подвох accuracy
В почте 100 писем, из них только 5 - спам. «Модель» всегда отвечает `'не спам'`.

In [28]:
y_true = ['не спам'] * 95 + ['спам'] * 5
y_pred = ['не спам'] * 100

print("Accuracy ленивой модели:", accuracy(y_true, y_pred))

Accuracy ленивой модели: 0.95


**Обсудим:** accuracy = 95%, а модель не нашла **ни одного** спам-письма. Хорошая ли это модель? Когда accuracy обманывает?

## 4. Расстояние между объектами

KNN ищет **похожие** объекты. Похожесть измеряем расстоянием. Для двух признаков это теорема Пифагора:
$$\rho(A, B) = \sqrt{(a_1 - b_1)^2 + (a_2 - b_2)^2}$$

Для любого числа признаков $d$ - то же самое, просто слагаемых больше:
$$\rho(A, B) = \sqrt{\sum_{j=1}^{d}(a_j - b_j)^2}$$

Пример: $A = (1; 2)$, $B = (4; 6)$, тогда $\rho = \sqrt{3^2 + 4^2} = \sqrt{25} = 5$.

### Задание 3. Функция distance
Функция должна работать для списков **любой длины**: пройдите циклом по признакам, сложите квадраты разностей и возьмите корень `math.sqrt`.

In [29]:
def distance(a, b):
    total = 0
    for i in range(len(a)):
        total += (a[i] - b[i]) ** 2
    return math.sqrt(total)

In [30]:
try:
    assert distance([1, 2], [4, 6]) == 5, "расстояние между (1; 2) и (4; 6) должно быть 5"
    assert distance([0, 0, 0], [1, 2, 2]) == 3, "для трёх признаков: √(1 + 4 + 4) = 3"
    assert distance([7, 7], [7, 7]) == 0, "расстояние от точки до самой себя - 0"
    print("Верно!")
except Exception as e:
    print("Не верно(", e)

Верно!


### Ответ к заданию 3
Найдите расстояние между **самой первой** квартирой выборки `X_flats[0]` и квартирой `X_flats[4]` (80 м², 15 этаж).

**Запишите ответ - число, округлённое до 2 знаков после запятой.**

In [31]:
answer_3 = distance(X_flats[0], X_flats[4])
print("Ответ 3:", round(answer_3, 2))

Ответ 3: 39.61


## 5. KNN своими руками

**Алгоритм для нового объекта `x`:**
1. посчитать расстояние от `x` до **каждого** объекта обучающей выборки;
2. отсортировать объекты по расстоянию;
3. взять `k` ближайших;
4. **регрессия** - среднее их ответов, **классификация** - самый частый класс.

Сначала напишем общую часть - поиск соседей.

In [32]:
def get_neighbors(X_train, y_train, x, k):
    # 1. Считаем расстояния: список пар (расстояние, ответ)
    pairs = []
    for i in range(len(X_train)):
        d = distance(X_train[i], x)
        pairs.append((d, y_train[i]))

    # 2. Сортируем по расстоянию - по первому элементу пары
    pairs = sorted(pairs, key=lambda pair: pair[0])

    # 3. Берём k ближайших
    return pairs[:k]


# Посмотрим на соседей первой тестовой квартиры
x = X_flats_test[0]
print("Новая квартира:", x)
for d, price in get_neighbors(X_flats_train, y_flats_train, x, k=3):
    print(f"  сосед на расстоянии {d:5.2f}, цена {price}")

Новая квартира: [45, 6, 9]
  сосед на расстоянии  3.32, цена 8.1
  сосед на расстоянии  4.69, цена 8.8
  сосед на расстоянии  9.70, цена 6.9


### KNN для регрессии
Прогноз - среднее ответов $k$ соседей:
$$a(x) = \frac{1}{k}\sum_{j=1}^{k} y_{(j)}$$

In [33]:
def knn_regression(X_train, y_train, x, k):
    neighbors = get_neighbors(X_train, y_train, x, k)
    total = 0
    for d, answer in neighbors:
        total += answer
    return total / k


predictions = []
for x in X_flats_test:
    predictions.append(knn_regression(X_flats_train, y_flats_train, x, k=3))

for x, p, t in zip(X_flats_test, predictions, y_flats_test):
    print(f"квартира {x}: прогноз {p:5.2f}, правда {t}")
print()
print(f"MAE на тесте: {mae(y_flats_test, predictions):.3f} млн руб.")

квартира [45, 6, 9]: прогноз  7.93, правда 8.6
квартира [60, 10, 7]: прогноз 11.90, правда 11.5
квартира [35, 1, 20]: прогноз  6.87, правда 6.0
квартира [75, 14, 3]: прогноз 13.50, правда 14.6

MAE на тесте: 0.758 млн руб.


### Задание 4. KNN для классификации
Ответ - **самый частый класс** среди соседей. Посчитайте голоса в словаре `votes`: ключ - класс, значение - сколько соседей за него.

Подсказка: `votes.get(label, 0)` возвращает текущее число голосов (или 0, если класса ещё нет в словаре).

In [34]:
def knn_classify(X_train, y_train, x, k):
    neighbors = get_neighbors(X_train, y_train, x, k)
    votes = {}
    for d, label in neighbors:
        if votes.get(label, 0) == 0:
            votes[label] = 1
        else:
            votes[label] += 1
    # Класс с максимальным числом голосов:
    return max(votes, key=votes.get)

In [35]:
try:
    assert knn_classify([[0], [1], [10]], ['a', 'a', 'b'], [0.5], k=3) == 'a', "два соседа 'a' против одного 'b' - ответ 'a'"
    assert knn_classify([[0], [1], [10]], ['a', 'a', 'b'], [9], k=1) == 'b', "при k = 1 ответ - класс ближайшего соседа"
    print("Верно!")
except Exception as e:
    print("Не верно(", e)

Верно!


In [36]:
# Классифицируем тестовые фрукты
predictions = []
for x in X_fruits_test:
    predictions.append(knn_classify(X_fruits_train, y_fruits_train, x, k=3))

for x, p, t in zip(X_fruits_test, predictions, y_fruits_test):
    mark = "YES" if p == t else "NO"
    print(f"{mark} фрукт {x}: прогноз «{p}», правда «{t}»")
print()
print("Accuracy на тесте:", accuracy(y_fruits_test, predictions))

YES фрукт [152, 9]: прогноз «яблоко», правда «яблоко»
YES фрукт [180, 4]: прогноз «груша», правда «груша»
YES фрукт [168, 8]: прогноз «яблоко», правда «яблоко»
YES фрукт [162, 4]: прогноз «груша», правда «груша»

Accuracy на тесте: 1.0


## 6. Эксперимент: какое k выбрать?

Переберём $k$ от 1 до 7 и посмотрим на качество на тесте. Полоска из `█` - наш «график» без библиотек.

In [37]:
print("Квартиры (MAE - чем меньше, тем лучше)")
for k in range(1, 8):
    preds = [knn_regression(X_flats_train, y_flats_train, x, k) for x in X_flats_test]
    err = mae(y_flats_test, preds)
    print(f"k = {k}: MAE = {err:5.2f}  " + "|" * round(err * 10))

print()
print("Фрукты (accuracy - чем больше, тем лучше)")
for k in range(1, 8):
    preds = [knn_classify(X_fruits_train, y_fruits_train, x, k) for x in X_fruits_test]
    acc = accuracy(y_fruits_test, preds)
    print(f"k = {k}: accuracy = {acc:4.2f}  " + "|" * round(acc * 20))

Квартиры (MAE - чем меньше, тем лучше)
k = 1: MAE =  0.65  |||||||
k = 2: MAE =  0.29  |||
k = 3: MAE =  0.76  ||||||||
k = 4: MAE =  0.94  |||||||||
k = 5: MAE =  1.57  ||||||||||||||||
k = 6: MAE =  1.58  ||||||||||||||||
k = 7: MAE =  2.20  ||||||||||||||||||||||

Фрукты (accuracy - чем больше, тем лучше)
k = 1: accuracy = 0.75  |||||||||||||||
k = 2: accuracy = 0.75  |||||||||||||||
k = 3: accuracy = 1.00  ||||||||||||||||||||
k = 4: accuracy = 0.75  |||||||||||||||
k = 5: accuracy = 0.50  ||||||||||
k = 6: accuracy = 0.75  |||||||||||||||
k = 7: accuracy = 0.75  |||||||||||||||


**Обсудим:**
* Какое $k$ лучше для квартир? Почему при $k = 1$ результат хуже?
* Что будет при $k = 8$, то есть когда соседи - вся обучающая выборка?
* Честно ли выбирать $k$ по тестовой выборке? (Подсказка: для этого заводят отдельную **валидационную** выборку.)

In [ ]:
def mape(y_true, y_pred):
    total = 0
    for i in range(len(y_true)):
        total += abs(y_true[i] - y_pred[i]) / abs(y_true[i])
    return 100 * total / (len(y_true))

In [1]:
def normalize(X):
    mins = []
    maxs = []
    new_X = X.copy()
    for j in range(len(X[0])):
        mx = X[0][j]
        mn = X[0][j]
        for i in range(len(X)):
            mx = max(mx, X[i][j])
            mn = min(mn, X[i][j])
        mins.append(mn)
        maxs.append(mx)
    for j in range(len(X[0])):
        for i in range(len(X)):
            if mins[j] == maxs[j]:
                new_X[i][j] = 0
            else:
                new_X[i][j] = (X[i][j] - mins[j]) / (maxs[j] - mins[j])
    return new_X

[[0, 0.10365175746172924, 1.0, 0.0], [0, 1.0, 0.0, 1.0], [0, 0.0, 0.3161026403867609, 0.9911504424778761]]


In [ ]:
def manhattan(a, b):
    total = 0
    for i in range(len(a)):
        total += abs(a[i] - b[i])
    return total